In [1]:
import cv2
import numpy as np

In [2]:
def compute_and_visualize_flow(video_path, output_path, max_seconds=30):
    """
    Computes dense optical flow for a video and saves the visualization.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video stream or file: {video_path}")
        return

    # Get video properties
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Optional: Downscale resolution by half to make processing much faster
    new_w, new_h = width // 2, height // 2

    # Define the codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (new_w, new_h))

    # Read the first frame and convert to grayscale
    ret, frame1 = cap.read()
    if not ret:
        print("Failed to read the first frame.")
        return

    prvs = cv2.resize(cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY), (new_w, new_h))

    # Create an HSV image array to store the visual representation
    hsv = np.zeros((new_h, new_w, 3), dtype=np.float32)
    hsv[..., 1] = 255  # Set saturation to maximum for vibrant colors

    frame_count = 0
    max_frames = int(fps * max_seconds)

    print(f"Processing optical flow for {video_path}...")

    while cap.isOpened() and frame_count < max_frames:
        ret, frame2 = cap.read()
        if not ret:
            break

        next_frame = cv2.resize(cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY), (new_w, new_h))

        # Calculate Dense Optical Flow using Gunnar Farneback's algorithm
        flow = cv2.calcOpticalFlowFarneback(prvs, next_frame, None,
                                            pyr_scale=0.5, levels=3, winsize=15,
                                            iterations=3, poly_n=5, poly_sigma=1.2, flags=0)

        # Convert the flow vectors (u, v) into polar coordinates (magnitude, angle)
        mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])

        # Map the angle to Hue (direction of motion)
        # OpenCV's Hue ranges from 0 to 179
        hsv[..., 0] = ang * 180 / np.pi / 2

        # Map the magnitude to Value (speed of motion)
        hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)

        # Convert the HSV image back to BGR format to write to the video file
        bgr_flow = cv2.cvtColor(np.uint8(hsv), cv2.COLOR_HSV2BGR)
        out.write(bgr_flow)

        # Update the previous frame
        prvs = next_frame
        frame_count += 1

    cap.release()
    out.release()
    print(f"Success! Visualized video saved to: {output_path}")

In [3]:
compute_and_visualize_flow("/content/IMG_1686.MOV", "optical_flow_output1.mp4", max_seconds=30)

Processing optical flow for /content/IMG_1686.MOV...
Success! Visualized video saved to: optical_flow_output1.mp4


In [ ]:
compute_and_visualize_flow("/content/IMG_2794.MOV", "optical_flow_output2.mp4", max_seconds=30)

In [4]:
def bilinear_interpolate(image, x, y):
    """Calculates pixel intensity at a fractional (sub-pixel) coordinate."""
    x1, y1 = int(np.floor(x)), int(np.floor(y))
    x2, y2 = x1 + 1, y1 + 1

    # Boundary checks
    if x1 < 0 or x2 >= image.shape[1] or y1 < 0 or y2 >= image.shape[0]:
        return 0.0

    # Get the four surrounding integer pixel intensities
    I11 = float(image[y1, x1])
    I21 = float(image[y1, x2])
    I12 = float(image[y2, x1])
    I22 = float(image[y2, x2])

    # Fractional distances
    dx = x - x1
    dy = y - y1

    # Interpolate in X
    R1 = (1 - dx) * I11 + dx * I21
    R2 = (1 - dx) * I12 + dx * I22

    # Interpolate in Y
    P = (1 - dy) * R1 + dy * R2
    return P

def validate_tracking(video_path, frame_idx, track_x, track_y, window_size=5):
    """Sets up LK tracking for two frames and validates via interpolation."""
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)

    ret1, frame1 = cap.read()
    ret2, frame2 = cap.read()
    cap.release()

    if not ret1 or not ret2:
        print("Error reading frames.")
        return

    # Convert to grayscale floats for math
    f1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY).astype(np.float64)
    f2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY).astype(np.float64)

    # 1. Setup the localized window
    half_w = window_size // 2
    patch1 = f1[track_y-half_w : track_y+half_w+1, track_x-half_w : track_x+half_w+1]
    patch2 = f2[track_y-half_w : track_y+half_w+1, track_x-half_w : track_x+half_w+1]

    # 2. Calculate Gradients (Ix, Iy, It)
    # Using Sobel for spatial gradients, direct subtraction for temporal
    Ix = cv2.Sobel(patch1, cv2.CV_64F, 1, 0, ksize=3).flatten()
    Iy = cv2.Sobel(patch1, cv2.CV_64F, 0, 1, ksize=3).flatten()
    It = (patch2 - patch1).flatten()

    # 3. Setup and solve A^T A v = A^T b
    A = np.vstack((Ix, Iy)).T
    b = -It

    try:
        # Least squares solution for (u, v)
        nu = np.linalg.pinv(A.T @ A) @ A.T @ b
        u, v = nu[0], nu[1]
    except np.linalg.LinAlgError:
        print("Singular matrix: Choose a coordinate with better texture/corners.")
        return

    # 4. Validation
    new_x = track_x + u
    new_y = track_y + v

    original_intensity = f1[track_y, track_x]
    interpolated_intensity = bilinear_interpolate(f2, new_x, new_y)

    print(f"--- Tracking Validation Results ---")
    print(f"Original Coordinate (Frame 1):  ({track_x}, {track_y})")
    print(f"Calculated Motion (u, v):       ({u:.3f}, {v:.3f})")
    print(f"New Sub-pixel Coord (Frame 2):  ({new_x:.3f}, {new_y:.3f})\n")

    print(f"Intensity at Original Coord:    {original_intensity:.2f}")
    print(f"Intensity at Interpolated Coord:{interpolated_intensity:.2f}")
    print(f"Difference (Error):             {abs(original_intensity - interpolated_intensity):.2f}")

In [5]:
validate_tracking("/content/IMG_1686.MOV", frame_idx=150, track_x=40, track_y=1000, window_size=5)

--- Tracking Validation Results ---
Original Coordinate (Frame 1):  (40, 1000)
Calculated Motion (u, v):       (0.548, -0.495)
New Sub-pixel Coord (Frame 2):  (40.548, 999.505)

Intensity at Original Coord:    7.00
Intensity at Interpolated Coord:7.74
Difference (Error):             0.74


In [17]:
validate_tracking("/content/IMG_2794.MOV", frame_idx=125, track_x=1105, track_y=500, window_size=5)

--- Tracking Validation Results ---
Original Coordinate (Frame 1):  (1105, 500)
Calculated Motion (u, v):       (-29.240, -28.052)
New Sub-pixel Coord (Frame 2):  (1075.760, 471.948)

Intensity at Original Coord:    198.00
Intensity at Interpolated Coord:198.95
Difference (Error):             0.95
